# Ontology Pre/Post Evaluation

This notebook reproduces the scenario preparation and paired comparison between the ontology before enrichment (`HLO_T1.owl`) and after enrichment (`HLO_Template_LLM.owl`).

The evaluation set is constructed in three stages:

- 300 parent-placement scenarios from shared MIMO-source concepts.
- 200 non-overlapping reference-lookup scenarios from shared concepts with post-enrichment reference codes.
- A single combined scenario file used unchanged for both ontology releases.

All selections are deterministic and retain the original `sourceIdentifier` for traceability.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd()
EVAL = cwd if (cwd / "generate_parent.py").exists() else cwd / "evaluation"
if not EVAL.exists():
    raise FileNotFoundError("Run this notebook from the project root or the evaluation folder.")

sys.path.insert(0, str(EVAL))

from generate_parent import generate as generate_parent
from generate_reference import generate as generate_reference
from merge_scenarios import merge
from compare_ontologies import compare

BEFORE = EVAL / "ontologies" / "before" / "HLO_T1.owl"
AFTER = EVAL / "ontologies" / "after" / "HLO_Template_LLM.owl"
DATA = EVAL / "data"
RESULTS = EVAL / "results"

PARENT = DATA / "parent_scenarios.csv"
REFERENCE = DATA / "reference_scenarios.csv"
SCENARIOS = DATA / "scenarios.csv"

BEFORE, AFTER

(PosixPath('/mnt/data/hlo_minimal/evaluation/ontologies/before/HLO_T1.owl'),
 PosixPath('/mnt/data/hlo_minimal/evaluation/ontologies/after/HLO_Template_LLM.owl'))

## 1. Parent-placement scenarios

The parent set is sampled from concepts that:

1. belong to the MIMO-source group in both ontology releases;
2. have the same persistent `sourceIdentifier` in both releases;
3. have a label and a direct parent in the pre-enrichment ontology.

A fixed seed is used so the same 300 concepts are selected on every run.

In [2]:
parent_rows = generate_parent(BEFORE, AFTER, PARENT, count=300, seed=73)
print("Parent scenarios:", len(parent_rows))
pd.read_csv(PARENT).head()

Parent scenarios: 300


,scenario_id,scenario_type,source_identifier,query,expected_label,expected_category,expected_parent,selection_seed
0,P001,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1,Alert,Alert,base-concept,MIMOConcept,73
1,P002,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#10,Biomedical Informatics,Biomedical Informatics,base-concept,MIMOConcept,73
2,P003,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,data display,data display,base-concept,MIMOConcept,73
3,P004,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,computer-aided design,computer-aided design,base-concept,MIMOConcept,73
4,P005,parent_placement,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,workflow,workflow,base-concept,MIMOConcept,73


## 2. Reference-lookup scenarios

The reference set uses concepts that are shared by the two releases, do not overlap with the parent set, and have a `conceptCode` in the post-enrichment ontology. The code becomes the lookup query while `sourceIdentifier` remains the stable expected identity.

A second fixed seed selects 200 records reproducibly.

In [3]:
reference_rows = generate_reference(BEFORE, AFTER, PARENT, REFERENCE, count=200, seed=74)
print("Reference scenarios:", len(reference_rows))
pd.read_csv(REFERENCE).head()

Reference scenarios: 200


,scenario_id,scenario_type,source_identifier,query,expected_label,expected_category,expected_concept_code,selection_seed
0,R001,reference_lookup,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,H517,information science,base-concept,H517,74
1,R002,reference_lookup,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,H773,"Printing, Three-Dimensional",base-concept,H773,74
2,R003,reference_lookup,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,H437,Complexity Analysis,base-concept,H437,74
3,R004,reference_lookup,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,H761,cybernetics,base-concept,H761,74
4,R005,reference_lookup,http://chu-rouen.fr/cismef/Dictionnaire_EFMI#1...,H840,library science,base-concept,H840,74


## 3. Combined evaluation set

The two non-overlapping sets are combined and validated. The resulting file contains exactly 500 scenarios and is the only scenario file used in the paired comparison.

In [4]:
combined_rows = merge(PARENT, REFERENCE, SCENARIOS)
scenario_df = pd.read_csv(SCENARIOS)
print("Combined scenarios:", len(combined_rows))
display(scenario_df.groupby("scenario_type").size().rename("count").to_frame())

Combined scenarios: 500


,count
scenario_type,
parent_placement,300
reference_lookup,200


## 4. Paired ontology comparison

The same scenario rows are evaluated against both releases.

- Parent-placement cases resolve the concept through `sourceIdentifier` and verify its direct parent.
- Reference-lookup cases resolve the query through `conceptCode` and verify that the resolved class has the expected `sourceIdentifier`.
- Structural counts are recorded alongside scenario-level results.

In [5]:
before_summary, after_summary, comparison = compare(BEFORE, AFTER, SCENARIOS, RESULTS)
summary_df = pd.DataFrame(comparison)
display(summary_df)

,metric,before,after,difference
0,triples,14651.0,17839.0,3188.0
1,classes,1642.0,1642.0,0.0
2,source_identifier_values,1593.0,1593.0,0.0
3,concept_code_values,0.0,1610.0,1610.0
4,parent_cases,300.0,300.0,0.0
5,parent_passed,300.0,300.0,0.0
6,parent_accuracy,1.0,1.0,0.0
7,reference_cases,200.0,200.0,0.0
8,reference_passed,0.0,200.0,200.0
9,reference_accuracy,0.0,1.0,1.0


## 5. Row-level evidence

The detailed files below preserve one result row per scenario for each ontology release. They can be inspected directly when auditing an individual case.

In [6]:
before_results = pd.read_csv(RESULTS / "before_results.csv")
after_results = pd.read_csv(RESULTS / "after_results.csv")

print("Before rows:", len(before_results))
print("After rows:", len(after_results))

display(before_results.groupby(["scenario_type", "passed"]).size().rename("count").to_frame())
display(after_results.groupby(["scenario_type", "passed"]).size().rename("count").to_frame())

Before rows: 500
After rows: 500


,,count
scenario_type,passed,
parent_placement,True,300
reference_lookup,False,200


,,count
scenario_type,passed,
parent_placement,True,300
reference_lookup,True,200


## Output files

Generated data are stored under `evaluation/data/` and paired evaluation results under `evaluation/results/`. The two ontology releases used by this notebook are kept under `evaluation/ontologies/before/` and `evaluation/ontologies/after/`.